# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows FAIR principles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Dataset
dataset = mlc.Dataset(url)
# Access the dataset metadata
metadata = dataset.metadata.to_json()

print("Dataset Title:", metadata.get('name', '<unknown>'))
print("Description:", metadata.get('description', '<no description>'))

# Print some metadata fields
print("Version:", metadata.get('version', '<no version info>'))
print("Published Date:", metadata.get('datePublished', '<no date info>'))
print("Citation:", metadata.get('citeAs', '<no citation info>'))

## 2. Data Overview
Review available record sets, fields, and their `@id`s (unique identifiers).
For Croissant datasets, each record set, field, column, and entity is identified with its `@id`. This ensures clarity and reproducibility.

In [ ]:
# Examine available record sets, fields, and their @id
overview = dataset.metadata.to_json()

# In Croissant, record sets are under the 'recordSet' key
record_sets = overview.get('recordSet', [])
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print("Available record sets (@id):")
    for rs in record_sets:
        if isinstance(rs, dict):
            print("  -", rs.get('@id', '<no id>'))
        else:
            print("  -", rs)

    # For each record set, print fields and columns (by @id)
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs
        print(f"\nRecord set @id: {rs_id}")
        # Fetch full record set details
        rs_obj = dataset.metadata.record_set(rs_id)
        rs_json = rs_obj.to_json()
        fields = rs_json.get('field', [])
        columns = rs_json.get('column', [])
        print("Fields (@id):")
        for f in fields:
            print("  -", f.get('@id', '<no id>') if isinstance(f, dict) else f)
        print("Columns (@id):")
        for c in columns:
            print("  -", c.get('@id', '<no id>') if isinstance(c, dict) else c)

## 3. Data Extraction
Load tabular data from each record set and reference all entities by their `@id`.
Below, we extract the records from each record set, using Pandas for further analysis.

In [ ]:
# Find available record set @id's
rs_overview = dataset.metadata.to_json()
record_sets = rs_overview.get('recordSet', [])
record_set_ids = []
for rs in record_sets:
    if isinstance(rs, dict):
        record_set_ids.append(rs['@id'])
    else:
        record_set_ids.append(rs)

if not record_set_ids:
    print("No record sets found.")
else:
    print("Record sets for extraction:", record_set_ids)

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
        print("Sample columns:", df.columns.tolist())
        print(df.head(2))
    else:
        print(f"No records returned for {record_set_id}")

# For demonstration, pick the first available record set
main_record_set = record_set_ids[0] if record_set_ids else None
if main_record_set and main_record_set in dataframes:
    print("Main record set columns:", dataframes[main_record_set].columns.tolist())
    dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps: filtering records, normalization of numeric fields, and grouping. All entities are referenced via their `@id`.
For this dataset (clinicopathological colorectal cancers), common numeric fields may include age, diagnosis interval, or anatomical variables.

In [ ]:
# Example EDA
import numpy as np

# Pick a numeric field (e.g., age, diagnosis interval, etc.)
df = dataframes.get(main_record_set)
if df is not None and not df.empty:
    # Find available numeric columns
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    print("Numeric fields (@id):", numeric_cols)

    # If none, try to infer likely fields (e.g., 'age', 'interval', etc.)
    if not numeric_cols:
        guessed_cols = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower()]
        print("Guessed numeric fields:", guessed_cols)
        numeric_cols = guessed_cols

    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Referenced by @id (column name)

        # Set a filter threshold
        threshold = 40
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()

        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping example: group by a key categorical field
        # Try to find a grouping field (e.g., 'sex', 'msi_status', etc.)
        group_fields = [col for col in df.columns if ('sex' in col.lower() or 'msi' in col.lower())]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using the `matplotlib` and `seaborn` libraries.
All fields used are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = dataframes.get(main_record_set)

if df is not None and not df.empty:
    # Try numeric and grouping fields
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    group_fields = [col for col in df.columns if ('sex' in col.lower() or 'msi' in col.lower())]

    if numeric_cols:
        x_field = numeric_cols[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[x_field], bins=10, kde=True)
        plt.title(f"Distribution of {x_field} (@id)")
        plt.xlabel(x_field)
        plt.ylabel("Frequency")
        plt.show()

        # If group field exists, show grouped boxplot
        if group_fields:
            y_field = group_fields[0]
            plt.figure(figsize=(8, 5))
            sns.boxplot(x=df[y_field], y=df[x_field])
            plt.title(f"{x_field} by {y_field} (@id)")
            plt.xlabel(y_field)
            plt.ylabel(x_field)
            plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore a FAIR^2 clinical dataset using `mlcroissant`.
The approach ensures all entities are referenced by their unique `@id`, supporting transparency and reproducibility.

**Key Findings:**
- Dataset metadata was successfully loaded and reviewed.
- Data overview and record sets/fields identified by their `@id`.
- Example extraction, transformation, and visualization steps applied to clinical record sets.
- The notebook can be easily extended for advanced processing such as statistical modeling, deeper analysis, or machine learning workflows.

For further dataset curation or usage, consult the FAIR^2 schema or refer to `mlcroissant` documentation.